# 02 — Cross-study consistency check (Phase 2 hand-off to Phase 3)

Loads every `data/raw/{study_id}.h5ad` produced by `scripts/download_phase2.py`,
surfaces per-study summaries, and flags outliers that would otherwise
propagate silently into harmonization (Phase 3).

Per-study columns checked:
- `n_cells`, `n_donors`
- `donor_disease_status` ratio (diseased : healthy)
- `virus` distribution
- top cell types by count
- donor count per disease group (catches studies w/ 1 healthy donor → no real control)

**Outlier criteria** (any one triggers a flag; review before harmonization):
- diseased:healthy ratio outside [0.25, 4]
- < 3 donors in either group
- top-3 cell type composition wildly different from cohort median

Pure laptop CPU, seconds to run.

In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

RAW_DIR = Path("..") / "data" / "raw"
h5ads = sorted(RAW_DIR.glob("*.h5ad"))
print(f"Found {len(h5ads)} h5ad files in {RAW_DIR.resolve()}")
for p in h5ads:
    print(f"  - {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

## Per-study summary table

In [ ]:
rows = []
for p in h5ads:
    a = ad.read_h5ad(p, backed="r")
    obs = (
        a.obs[["virus", "donor_disease_status", "donor_id", "cell_type"]].to_pandas()
        if hasattr(a.obs, "to_pandas")
        else a.obs[["virus", "donor_disease_status", "donor_id", "cell_type"]].copy()
    )

    n_diseased = int((obs["donor_disease_status"] == "diseased").sum())
    n_healthy = int((obs["donor_disease_status"] == "healthy").sum())
    ratio = n_diseased / max(n_healthy, 1)

    donors_diseased = int(obs.loc[obs["donor_disease_status"] == "diseased", "donor_id"].nunique())
    donors_healthy = int(obs.loc[obs["donor_disease_status"] == "healthy", "donor_id"].nunique())

    rows.append(
        {
            "study_id": p.stem,
            "n_cells": a.n_obs,
            "n_donors_total": int(obs["donor_id"].nunique()),
            "n_donors_diseased": donors_diseased,
            "n_donors_healthy": donors_healthy,
            "n_cells_diseased": n_diseased,
            "n_cells_healthy": n_healthy,
            "ratio_d_to_h": round(ratio, 2),
            "viruses": ",".join(sorted(set(obs["virus"].astype(str)) - {"mock"})),
            "n_cell_types": int(obs["cell_type"].nunique()),
        }
    )

summary = pd.DataFrame(rows)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)
summary

## Outlier flags

In [ ]:
flags = []
for _, row in summary.iterrows():
    issues = []
    if row["ratio_d_to_h"] < 0.25 or row["ratio_d_to_h"] > 4:
        issues.append(f"diseased:healthy cell ratio {row['ratio_d_to_h']} outside [0.25, 4]")
    if row["n_donors_diseased"] < 3:
        issues.append(f"only {row['n_donors_diseased']} diseased donors")
    if row["n_donors_healthy"] < 3:
        issues.append(f"only {row['n_donors_healthy']} healthy donors")
    if issues:
        flags.append({"study_id": row["study_id"], "issues": "; ".join(issues)})

if flags:
    print("FLAGGED STUDIES (review before harmonization):")
    print(pd.DataFrame(flags).to_string(index=False))
else:
    print("No outliers flagged on basic donor-count / ratio criteria.")

## Top cell types per study
Detects wildly skewed cell-type compositions (e.g. one study is 90% T cells, another is 90% monocytes — Harmony will struggle).

In [ ]:
top_n = 5
for p in h5ads:
    a = ad.read_h5ad(p, backed="r")
    counts = a.obs["cell_type"].value_counts(normalize=True).head(top_n)
    print(f"\n=== {p.stem} (top {top_n} cell types) ===")
    for ct, frac in counts.items():
        print(f"  {frac * 100:5.1f}%  {ct}")

## Cell-type vocabulary union — harmonization difficulty preview
Number of unique cell-type strings across all studies. Larger = more harmonization work in Phase 3 (Cell Ontology mapping).

In [ ]:
all_types = set()
per_study_types = {}
for p in h5ads:
    a = ad.read_h5ad(p, backed="r")
    types = set(a.obs["cell_type"].astype(str))
    per_study_types[p.stem] = types
    all_types |= types

print(f"Union of cell-type labels across all studies: {len(all_types)} unique strings")
print()
for sid, types in per_study_types.items():
    print(f"  {sid:<28} {len(types):>4} unique cell types")

# Cell types in only ONE study — likely study-specific labelling, will need harmonization
study_membership = pd.DataFrame(
    {sid: [t in s for t in all_types] for sid, s in per_study_types.items()},
    index=sorted(all_types),
)
n_studies_per_type = study_membership.sum(axis=1)
print(f"\nCell types present in only 1 study: {(n_studies_per_type == 1).sum()}")
print(
    f"Cell types present in all {len(per_study_types)} studies: {(n_studies_per_type == len(per_study_types)).sum()}"
)

## Cross-study within-virus response check (PBMC SARS-CoV-2 only)
Pre-Harmony reference. Compute response vector (mean(diseased) - mean(healthy)) per SARS-CoV-2 PBMC study, then pairwise Pearson r between studies. This is the floor; post-Harmony correlations should not collapse far below the within-study baseline OR rise far above ~0.7 (PLAN Phase 3 watchpoint).

In [ ]:
import scanpy as sc


def quick_response(adata):
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    d = adata[adata.obs["donor_disease_status"] == "diseased"].X
    h = adata[adata.obs["donor_disease_status"] == "healthy"].X
    return pd.Series(
        np.asarray(d.mean(axis=0)).ravel() - np.asarray(h.mean(axis=0)).ravel(),
        index=adata.var_names,
    )


rvs = {}
for p in h5ads:
    a = ad.read_h5ad(p)
    # Restrict to SARS-CoV-2 cells (+ mock); if a study has IAV cells (e.g. Lee), drop them here
    sars_mask = a.obs["virus"].isin(["sars_cov_2", "mock"])
    if (
        sars_mask.sum() < 100
        or (a.obs.loc[sars_mask, "donor_disease_status"].value_counts() < 50).any()
    ):
        print(f"  skipping {p.stem} (insufficient SARS-CoV-2 + healthy cells)")
        continue
    sub = a[sars_mask].copy()
    rvs[p.stem] = quick_response(sub)

studies = list(rvs.keys())
print(f"\nResponse vectors computed for {len(studies)} SARS-CoV-2 PBMC studies")

if len(studies) >= 2:
    shared = rvs[studies[0]].index
    for s in studies[1:]:
        shared = shared.intersection(rvs[s].index)
    aligned = pd.DataFrame({s: rvs[s].loc[shared] for s in studies})
    corr = aligned.corr(method="pearson")
    print(f"\nShared genes: {len(shared)}")
    print("\nPairwise Pearson r (response vectors, pre-Harmony):")
    print(corr.round(3))
    print()
    print(f"Mean off-diagonal r: {(corr.values[~np.eye(len(corr), dtype=bool)]).mean():.3f}")